In [1]:
import torch
import cv2
import numpy as np
import time

def process_with_pytorch(input_filename, output_filename):
    # 1. Check Device
    #if torch.cuda.is_available():
    device = torch.device("cuda")
    #    print(f"✅ PyTorch found GPU: {torch.cuda.get_device_name(0)}")
    #else:
    #    device = torch.device("cpu")
    #    print("⚠️ GPU not found. Running on CPU.")

    # 2. Read Image (OpenCV)
    img_bgr = cv2.imread(input_filename)
    if img_bgr is None:
        print("Error: Image not found.")
        return

    # Convert to RGB
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 3. Move to GPU (H2D)
    # Convert numpy array to PyTorch Tensor
    # Shape becomes (Height, Width, Channels)
    tensor_rgb = torch.from_numpy(img_rgb).float().to(device)

    print("Processing on GPU...")
    start_time = time.time()

    # 4. The Math (Tensor Operations)
    # PyTorch handles the parallelization automatically
    r = tensor_rgb[:, :, 0]
    g = tensor_rgb[:, :, 1]
    b = tensor_rgb[:, :, 2]

    gray = (0.299 * r) + (0.587 * g) + (0.114 * b)

    # Force synchronization to get accurate timing (GPU is async)
    torch.cuda.synchronize()
    end_time = time.time()

    print(f"Done in {end_time - start_time:.5f} seconds.")

    # 5. Save Result (D2H)
    # Move back to CPU and convert to uint8
    result_cpu = gray.cpu().numpy().astype(np.uint8)

    cv2.imwrite(output_filename, result_cpu)
    print(f"Saved to {output_filename}")

if __name__ == "__main__":
    process_with_pytorch("p.jpg", "output_torch.jpg")

Processing on GPU...
Done in 0.06692 seconds.
Saved to output_torch.jpg


In [5]:
import torch
import cv2
import numpy as np
import time
import os

def run_benchmark(input_filename):
    # 0. Load Image (Common Step)
    if not os.path.exists(input_filename):
        print(f"Error: {input_filename} not found.")
        return

    #print(f"Loading {input_filename}...")
    img_bgr = cv2.imread(input_filename)
    
    # Convert BGR to RGB
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    # Create Base Tensor (Float32)
    # This sits in RAM initially
    base_tensor = torch.from_numpy(img_rgb).float()
    
    h, w, c = base_tensor.shape
    print(f"Image Size: {w}x{h} pixels")
    #print("-" * 40)

    # --- 1. CPU TEST ---
    print("Running on CPU...", end="", flush=True)
    
    # Move data to CPU (Explicitly)
    device_cpu = torch.device("cpu")
    input_cpu = base_tensor.to(device_cpu)
    
    start_cpu = time.time()
    
    # The Math
    r = input_cpu[:, :, 0]
    g = input_cpu[:, :, 1]
    b = input_cpu[:, :, 2]
    gray_cpu = (0.299 * r) + (0.587 * g) + (0.114 * b)
    
    end_cpu = time.time()
    cpu_time = end_cpu - start_cpu
    print(f"Time: {cpu_time:.5f} seconds")

    # --- 2. GPU TEST ---
    if torch.cuda.is_available():
        print("Running on GPU...", end="", flush=True)
        device_gpu = torch.device("cuda")
        
        # Move data to GPU (H2D Transfer)
        # Note: We include transfer time in real-world benchmarks usually, 
        # but here we focus on calculation speed.
        input_gpu = base_tensor.to(device_gpu)
        
        # WARMUP (Crucial for PyTorch/CUDA)
        # The first run initializes the CUDA context, which takes time.
        # We run a dummy calculation to "wake up" the GPU.
        _ = (0.299 * input_gpu[:, :, 0]) + (0.587 * input_gpu[:, :, 1])
        torch.cuda.synchronize()

        # REAL TIMING
        start_gpu = time.time()
        
        # The Math
        r = input_gpu[:, :, 0]
        g = input_gpu[:, :, 1]
        b = input_gpu[:, :, 2]
        gray_gpu = (0.299 * r) + (0.587 * g) + (0.114 * b)
        
        # Wait for GPU to finish (Async check)
        torch.cuda.synchronize()
        
        end_gpu = time.time()
        gpu_time = end_gpu - start_gpu
        print(f"Time: {gpu_time:.5f} seconds")

    
        result_final = gray_gpu.cpu().numpy().astype(np.uint8)
        cv2.imwrite("output_benchmark.jpg", result_final)
        
    else:
        print("⚠️ No GPU detected. Skipping GPU test.")

if __name__ == "__main__":
    # Make sure you have an image file here
    run_benchmark("p.jpg")

Image Size: 7680x4320 pixels
Running on CPU...Time: 0.22890 seconds
Running on GPU...Time: 0.01583 seconds
